# Evaluation — Satellite Image Classifier

Confusion matrix, per-class F1-score, and Grad-CAM attention maps
for model interpretability.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
from torchvision import models

## Evaluation Steps

1. Load best model checkpoint and test data
2. Generate predictions on held-out test set
3. Plot confusion matrix and per-class F1 scores
4. Implement Grad-CAM for attention visualization
5. Display Grad-CAM heatmaps on sample tiles

In [ ]:
# Load model and generate predictions
# model = torch.load('../models/resnet50_best.pth')
# model.eval()

# Placeholder: simulated predictions
np.random.seed(42)
class_names = ['agriculture', 'forest', 'urban', 'water', 'barren', 'grassland']
n_test = 500
y_true = np.random.randint(0, len(class_names), n_test)
y_pred = y_true.copy()
noise_idx = np.random.choice(n_test, size=50, replace=False)
y_pred[noise_idx] = np.random.randint(0, len(class_names), 50)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
fig, ax = plt.subplots(figsize=(8, 8))
disp.plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix — Satellite Image Classifier')
plt.tight_layout()
plt.show()

print(classification_report(y_true, y_pred, target_names=class_names))

In [ ]:
# Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, class_idx):
        output = self.model(input_tensor)
        self.model.zero_grad()
        output[0, class_idx].backward()
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = torch.relu(cam)
        cam = cam / (cam.max() + 1e-8)
        return cam.squeeze().cpu().numpy()

print('Grad-CAM class defined — ready for attention map generation')